# LeetCode 2198 — Number of Single Divisor Triplets bằng Python

Link đề: https://leetcode.com/problems/number-of-single-divisor-triplets/ · Độ khó: **Medium** · Chủ đề: Math, Counting, Enumeration

> **Bài Premium:** đề trong notebook được viết lại theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, nên đối chiếu lại câu chữ và ví dụ với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def singleDivisorTriplet(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc đúng đề: đếm **bộ ba chỉ số có thứ tự** `(i, j, k)` đôi một khác nhau, và "chia hết cho **đúng một** trong `nums[i], nums[j], nums[k]`" là đếm theo **vị trí** (hai vị trí cùng giá trị được đếm hai lần).
2. Nhận ra điểm mấu chốt về ràng buộc: `n` tới `10⁵` nhưng **giá trị chỉ từ 1 tới 100** ⇒ đổi vòng lặp từ **chỉ số** sang **giá trị** (`Counter`).
3. Đếm số bộ chỉ số ứng với một bộ giá trị bằng **quy tắc nhân có trừ trùng**: `c[a]·(c[b] − [a=b])·(c[c] − [c=a] − [c=b])`.
4. Tối ưu thêm bằng cách chỉ duyệt bộ giá trị **không giảm** `a ≤ b ≤ c` và nhân với số hoán vị.
5. Kiểm thử đối chiếu brute force `O(n³)` với các cách đếm theo giá trị trên dữ liệu ngẫu nhiên có nhiều giá trị lặp (seed cố định).

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums` (đánh số từ 0). Bộ ba **chỉ số đôi một khác nhau** `(i, j, k)` gọi là **bộ ba một ước** nếu tổng `nums[i] + nums[j] + nums[k]` **chia hết cho đúng một** trong ba số `nums[i]`, `nums[j]`, `nums[k]`.

Trả về số bộ ba một ước. (Các bộ là **có thứ tự**: `(0, 3, 4)` và `(3, 0, 4)` khác nhau.)

**Ví dụ 1:** `nums = [4,6,7,3,2]` → `12`.
- Giá trị `{4, 3, 2}`: tổng `9` chỉ chia hết cho `3` ⇒ 6 hoán vị chỉ số.
- Giá trị `{4, 7, 3}`: tổng `14` chỉ chia hết cho `7` ⇒ 6 hoán vị.

**Ví dụ 2:** `nums = [1,2,2]` → `6`: tổng `5` chỉ chia hết cho `1`.

**Ví dụ 3:** `nums = [1,1,1]` → `0`: tổng `3` chia hết cho cả ba số.

**Ràng buộc:** `3 ≤ n ≤ 10⁵`, `1 ≤ nums[i] ≤ 100`.

In [1]:
import random
import time
from collections import Counter
from itertools import permutations, combinations, combinations_with_replacement, product
from math import comb
from typing import List

examples = [([4, 6, 7, 3, 2], 12), ([1, 2, 2], 6), ([1, 1, 1], 0)]
def single(a, b, c):
    """Tổng a+b+c chia hết cho đúng MỘT trong ba VỊ TRÍ a, b, c."""
    s = a + b + c
    return (s % a == 0) + (s % b == 0) + (s % c == 0) == 1

for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)
print("single(4,3,2) =", single(4, 3, 2), "| single(1,1,1) =", single(1, 1, 1))

[4, 6, 7, 3, 2] -> kỳ vọng 12
[1, 2, 2] -> kỳ vọng 6
[1, 1, 1] -> kỳ vọng 0
single(4,3,2) = True | single(1,1,1) = False


## 3. Kiến thức nền

### 3.1 Cộng giá trị boolean

Trong Python `True == 1`, `False == 0`, nên `(s % a == 0) + (s % b == 0) + (s % c == 0)` là **số vị trí** chia hết tổng. "Đúng một" ⇔ tổng này bằng 1.

In [2]:
s = 9
flags = [s % v == 0 for v in (4, 3, 2)]
print(flags, "-> số ước trong bộ:", sum(flags))
print(True + True + False)

[False, True, False] -> số ước trong bộ: 1
2


### 3.2 Bộ có thứ tự: `permutations` vs. không thứ tự: `combinations`

`permutations(range(n), 3)` sinh `n(n−1)(n−2)` bộ chỉ số có thứ tự, đôi một khác nhau. `combinations(range(n), 3)` sinh `C(n, 3)` bộ không thứ tự `i < j < k`; mỗi bộ không thứ tự ứng với đúng `3! = 6` bộ có thứ tự.

In [3]:
n = 4
print(len(list(permutations(range(n), 3))), "=", n * (n - 1) * (n - 2), "| combinations:", len(list(combinations(range(n), 3))), "x 6 =", 6 * comb(n, 3))

24 = 24 | combinations: 4 x 6 = 24


### 3.3 `Counter` và đổi biến "chỉ số → giá trị"

Vì điều kiện chỉ phụ thuộc **giá trị**, ta đếm `c[v]` = số vị trí có giá trị `v`. Số giá trị khác nhau `K ≤ 100` dù `n` có thể là `10⁵`.

In [4]:
c = Counter([4, 6, 7, 3, 2, 4, 4])
print(c, "| K =", len(c))

Counter({4: 3, 6: 1, 7: 1, 3: 1, 2: 1}) | K = 5


### 3.4 Quy tắc nhân có trừ trùng

Số bộ **chỉ số có thứ tự, đôi một khác nhau** `(i, j, k)` với `nums[i] = a, nums[j] = b, nums[k] = c`:
- chọn `i`: `c[a]` cách;
- chọn `j`: `c[b]` cách, nhưng nếu `b = a` thì phải trừ đi vị trí `i` đã dùng ⇒ `c[b] − [a = b]`;
- chọn `k`: `c[c] − [c = a] − [c = b]`.

(Ký hiệu `[P]` = 1 nếu `P` đúng, 0 nếu sai.) Nếu một thừa số ≤ 0 thì không có bộ nào.

In [5]:
def ways(cnt, a, b, c_):
    w = cnt[a] * (cnt[b] - (a == b)) * (cnt[c_] - (c_ == a) - (c_ == b))
    return max(w, 0)

cnt = Counter([2, 2, 2, 5])
print("(2,2,5):", ways(cnt, 2, 2, 5), "| kiểm:", sum(1 for t in permutations(range(4), 3) if [ [2,2,2,5][x] for x in t ] == [2, 2, 5]))
print("(2,2,2):", ways(cnt, 2, 2, 2), "| (5,5,2):", ways(cnt, 5, 5, 2))

(2,2,5): 6 | kiểm: 6
(2,2,2): 6 | (5,5,2): 0


### 3.5 Số hoán vị của một bộ giá trị

Với bộ giá trị không thứ tự `{a, b, c}`: nếu ba giá trị khác nhau có `6` cách sắp; nếu hai bằng nhau có `3` cách; nếu cả ba bằng nhau có `1` cách. Ta sẽ dùng nhận xét này ở Cách 4.

In [6]:
for t in [(2, 3, 5), (2, 2, 5), (2, 2, 2)]:
    print(t, "->", len(set(permutations(t))), "hoán vị khác nhau")

(2, 3, 5) -> 6 hoán vị khác nhau
(2, 2, 5) -> 3 hoán vị khác nhau
(2, 2, 2) -> 1 hoán vị khác nhau


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `nums = [4,6,7,3,2]`. Có `C(5,3) = 10` tập ba chỉ số. Tính tay từng tập giá trị:

| Giá trị | Tổng | Chia hết cho | Số ước | Một ước? |
|---|---|---|---|---|
| 4,6,7 | 17 | — | 0 | ✗ |
| 4,6,3 | 13 | — | 0 | ✗ |
| 4,6,2 | 12 | 4, 6, 2 | 3 | ✗ |
| 4,7,3 | 14 | 7 | 1 | ✓ |
| 4,7,2 | 13 | — | 0 | ✗ |
| 4,3,2 | 9 | 3 | 1 | ✓ |
| 6,7,3 | 16 | — | 0 | ✗ |
| 6,7,2 | 15 | — | 0 | ✗ |
| 6,3,2 | 11 | — | 0 | ✗ |
| 7,3,2 | 12 | 3, 2 | 2 | ✗ |

Hai tập thỏa ⇒ `2 × 6 = 12` bộ có thứ tự ✓.

**Quan sát quan trọng:** điều kiện chỉ dựa vào **giá trị**, mà giá trị ≤ 100. Thay vì duyệt `C(10⁵, 3) ≈ 1,7·10¹⁴` tập chỉ số, ta duyệt tối đa `100³ = 10⁶` bộ giá trị và nhân với số cách chọn chỉ số.

In [7]:
nums = [4, 6, 7, 3, 2]
for t in combinations(range(len(nums)), 3):
    vals = [nums[x] for x in t]; s = sum(vals)
    divs = [v for v in vals if s % v == 0]
    print(f"{t} giá trị {vals} tổng {s:2d} chia hết cho {divs} -> {'MỘT ƯỚC' if len(divs) == 1 else ''}")

(0, 1, 2) giá trị [4, 6, 7] tổng 17 chia hết cho [] -> 
(0, 1, 3) giá trị [4, 6, 3] tổng 13 chia hết cho [] -> 
(0, 1, 4) giá trị [4, 6, 2] tổng 12 chia hết cho [4, 6, 2] -> 
(0, 2, 3) giá trị [4, 7, 3] tổng 14 chia hết cho [7] -> MỘT ƯỚC
(0, 2, 4) giá trị [4, 7, 2] tổng 13 chia hết cho [] -> 
(0, 3, 4) giá trị [4, 3, 2] tổng  9 chia hết cho [3] -> MỘT ƯỚC
(1, 2, 3) giá trị [6, 7, 3] tổng 16 chia hết cho [] -> 
(1, 2, 4) giá trị [6, 7, 2] tổng 15 chia hết cho [] -> 
(1, 3, 4) giá trị [6, 3, 2] tổng 11 chia hết cho [] -> 
(2, 3, 4) giá trị [7, 3, 2] tổng 12 chia hết cho [3, 2] -> 


## 5. Cách 1 — Brute Force / baseline: mọi bộ chỉ số có thứ tự

**Ý tưởng:** duyệt `permutations(range(n), 3)`, đếm bộ thỏa `single`.

In [8]:
def triplets_brute(nums):
    return sum(1 for i, j, k in permutations(range(len(nums)), 3) if single(nums[i], nums[j], nums[k]))

for nums, exp in examples:
    got = triplets_brute(nums); print(nums, "->", got); assert got == exp

[4, 6, 7, 3, 2] -> 12
[1, 2, 2] -> 6
[1, 1, 1] -> 0


**Độ phức tạp:** `O(n³)` — với `n = 10⁵` là `10¹⁵` bộ. Không thể. Space `O(1)`.

**Nút thắt:** rất nhiều bộ chỉ số có **cùng bộ giá trị** — ta kiểm tra lại cùng một điều kiện vô số lần.

In [9]:
rng = random.Random(0)
for n in (20, 40, 80):
    a = [rng.randint(1, 100) for _ in range(n)]
    t = time.perf_counter(); triplets_brute(a)
    print(f"n={n}: {time.perf_counter() - t:.3f}s (gấp đôi n -> ~x8)")

n=20: 0.001s (gấp đôi n -> ~x8)
n=40: 0.006s (gấp đôi n -> ~x8)
n=80: 0.048s (gấp đôi n -> ~x8)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (gom theo giá trị):** đáp án `= Σ_{(a,b,c) có thứ tự} [single(a,b,c)] · ways(a, b, c)`, với `ways` như mục 3.4. Chỉ cần duyệt `a, b, c` trong tập giá trị xuất hiện (`K ≤ 100`).

**Insight 2 (bộ ba giá trị bằng nhau không bao giờ thỏa):** nếu `a = b = c` thì tổng `3a` chia hết cho cả ba vị trí ⇒ số ước là 3 ⇒ loại.

**Insight 3 (hai giá trị bằng nhau):** nếu `a = b ≠ c` thì hai vị trí giá trị `a` hoặc **cùng** chia hết, hoặc **cùng** không ⇒ để "đúng một", phải là: `a` **không** chia hết tổng, `c` chia hết tổng.

**Quy tắc "nếu… thì…" (Cách 4, duyệt `a ≤ b ≤ c`):**
- **Nếu** `a < b < c` và `single` ⇒ cộng `6 · c[a] · c[b] · c[c]`.
- **Nếu** đúng hai giá trị bằng nhau (`x` lặp, `y` đơn) và `single` ⇒ cộng `3 · c[x]·(c[x]−1) · c[y]` (bằng `6 · C(c[x], 2) · c[y]`).
- **Nếu** `a = b = c` ⇒ không cộng.

**Invariant:** mỗi bộ chỉ số có thứ tự được đếm **đúng một lần** — tại bộ giá trị (có thứ tự) của nó trong Cách 3, hoặc tại dạng sắp xếp không giảm của nó trong Cách 4.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt bộ không thứ tự `i < j < k`, nhân 6

Giảm hằng số 6 lần so với Cách 1 nhưng vẫn `O(n³)`.

In [10]:
def triplets_combinations(nums):
    return 6 * sum(1 for i, j, k in combinations(range(len(nums)), 3) if single(nums[i], nums[j], nums[k]))

for nums, exp in examples:
    assert triplets_combinations(nums) == exp
print([triplets_combinations(nums) for nums, _ in examples])

[12, 6, 0]


**Độ phức tạp Cách 2:** `O(n³/6)` — vẫn quá chậm cho `n` lớn.

### 7.2 Cách 3 — Duyệt bộ giá trị **có thứ tự** + quy tắc nhân trừ trùng

**Giải thích từng đoạn:**
1. `cnt = Counter(nums)`, `vals` = các giá trị khác nhau.
2. Với mọi `a, b, c` trong `vals` (có thể trùng): nếu `single(a, b, c)` thì cộng `ways(cnt, a, b, c)`.
3. `ways` tự xử lý trường hợp trùng giá trị (thừa số có thể bằng 0 khi không đủ phần tử).

In [11]:
def triplets_values_ordered(nums):
    cnt = Counter(nums); vals = list(cnt)
    ans = 0
    for a in vals:
        for b in vals:
            for c_ in vals:
                if single(a, b, c_):
                    ans += ways(cnt, a, b, c_)
    return ans

for nums, exp in examples:
    assert triplets_values_ordered(nums) == exp
print([triplets_values_ordered(nums) for nums, _ in examples])

[12, 6, 0]


In [12]:
class SolutionOrdered:
    def singleDivisorTriplet(self, nums: List[int]) -> int:
        cnt = Counter(nums); vals = list(cnt); ans = 0
        for a in vals:
            for b in vals:
                for c in vals:
                    s = a + b + c
                    if (s % a == 0) + (s % b == 0) + (s % c == 0) == 1:
                        ans += cnt[a] * (cnt[b] - (a == b)) * (cnt[c] - (c == a) - (c == b))
        return ans

print([SolutionOrdered().singleDivisorTriplet(nums) for nums, _ in examples])

[12, 6, 0]


**Lưu ý về `ways` âm:** khi một bộ giá trị **không thể** đạt được (ví dụ `a = b` nhưng `cnt[a] = 1`), một thừa số bằng 0 nên tích bằng 0. Có thể nào tích **âm** không? Thừa số thứ hai `cnt[b] − 1 ≥ 0`; thừa số thứ ba `cnt[c] − 2` âm chỉ khi `a = b = c` với `cnt = 1`, nhưng lúc đó thừa số thứ hai đã bằng 0 ⇒ tích 0. Vậy công thức an toàn; hơn nữa `a = b = c` không bao giờ `single`.

**Độ phức tạp Cách 3:** đếm `O(n)` + duyệt `O(K³)` với `K ≤ 100` ⇒ tối đa `10⁶` lần kiểm. Space `O(K)`.

### 7.3 Cách 4 — Duyệt bộ giá trị **không giảm** `a ≤ b ≤ c` (bản nộp chính)

Giảm khoảng 6 lần số bộ cần kiểm (`combinations_with_replacement` của `K` giá trị: `C(K+2, 3) ≈ 1,7·10⁵`). Với mỗi bộ, số bộ chỉ số có thứ tự tính bằng mục 3.5 + quy tắc nhân.

In [13]:
def triplets_values_sorted(nums):
    cnt = Counter(nums); vals = sorted(cnt)
    ans = 0
    for a, b, c_ in combinations_with_replacement(vals, 3):
        if a == c_:                                   # cả ba bằng nhau: không bao giờ thỏa
            continue
        if not single(a, b, c_):
            continue
        if a < b < c_:
            ans += 6 * cnt[a] * cnt[b] * cnt[c_]
        elif a == b:                                  # a = b < c
            ans += 3 * cnt[a] * (cnt[a] - 1) * cnt[c_]
        else:                                         # a < b = c
            ans += 3 * cnt[b] * (cnt[b] - 1) * cnt[a]
    return ans

for nums, exp in examples:
    assert triplets_values_sorted(nums) == exp
print([triplets_values_sorted(nums) for nums, _ in examples])

[12, 6, 0]


**Độ phức tạp Cách 4:** `O(n + K³/6)`. Space `O(K)`.

### 7.4 Cách 5 — Mảng đếm cố định 1..100 (không cần `Counter`)

Giống Cách 3 nhưng dùng `cnt = [0]*101` và duyệt `a, b, c` trong `1..100` có `cnt > 0`. Thuận tiện khi dịch sang ngôn ngữ khác (Java/C++), và cho thấy chi phí **không phụ thuộc** `n`.

In [14]:
def triplets_array(nums):
    cnt = [0] * 101
    for v in nums:
        cnt[v] += 1
    present = [v for v in range(1, 101) if cnt[v]]
    ans = 0
    for a in present:
        for b in present:
            for c_ in present:
                s = a + b + c_
                if (s % a == 0) + (s % b == 0) + (s % c_ == 0) == 1:
                    ans += cnt[a] * (cnt[b] - (a == b)) * (cnt[c_] - (c_ == a) - (c_ == b))
    return ans

for nums, exp in examples:
    assert triplets_array(nums) == exp
print([triplets_array(nums) for nums, _ in examples])

[12, 6, 0]


**Độ phức tạp Cách 5:** `O(n + K³)`, Space `O(100)`.

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [1, 2, 2, 3, 3, 3]` (tự dựng; kiểm bằng brute force ở dưới). `cnt = {1:1, 2:2, 3:3}`.

| (a, b, c) | tổng | ước (theo vị trí) | single? | loại | cộng |
|---|---|---|---|---|---|
| (1,1,1), (2,2,2), (3,3,3) | 3a | cả ba | ✗ | a=b=c | 0 |
| (1,1,2) | 4 | 1, 1, 2 | ✗ (3 ước) | | 0 |
| (1,1,3) | 5 | 1, 1 | ✗ (2 ước) | | 0 |
| (1,2,2) | 5 | 1 | ✓ | b=c | 3·2·1·1 = 6 |
| (1,2,3) | 6 | 1, 2, 3 | ✗ | | 0 |
| (1,3,3) | 7 | 1 | ✓ | b=c | 3·3·2·1 = 18 |
| (2,2,3) | 7 | — | ✗ (0 ước) | | 0 |
| (2,3,3) | 8 | 2 | ✓ | b=c | 3·3·2·2 = 36 |

(Bộ `(1,1,x)` cần hai vị trí giá trị 1 nhưng `cnt[1] = 1` nên dù thỏa cũng có hệ số 0.)

Tổng `6 + 18 + 36 = 60` (trace dưới in đầy đủ).

In [15]:
def trace_sorted(nums):
    cnt = Counter(nums); vals = sorted(cnt); ans = 0
    print("cnt =", dict(sorted(cnt.items())))
    for a, b, c_ in combinations_with_replacement(vals, 3):
        s = a + b + c_
        nd = (s % a == 0) + (s % b == 0) + (s % c_ == 0)
        if a == c_:
            add, kind = 0, "a=b=c (luôn 3 ước)"
        elif not nd == 1:
            add, kind = 0, f"{nd} ước"
        elif a < b < c_:
            add, kind = 6 * cnt[a] * cnt[b] * cnt[c_], "khác nhau x6"
        elif a == b:
            add, kind = 3 * cnt[a] * (cnt[a] - 1) * cnt[c_], "a=b x3"
        else:
            add, kind = 3 * cnt[b] * (cnt[b] - 1) * cnt[a], "b=c x3"
        ans += add
        print(f"({a},{b},{c_}) tổng {s:2d}: {kind:20s} +{add:3d} -> ans={ans}")
    return ans

r = trace_sorted([1, 2, 2, 3, 3, 3])
print("Đáp án:", r, "| brute force:", triplets_brute([1, 2, 2, 3, 3, 3]))

cnt = {1: 1, 2: 2, 3: 3}
(1,1,1) tổng  3: a=b=c (luôn 3 ước)   +  0 -> ans=0
(1,1,2) tổng  4: 3 ước                +  0 -> ans=0
(1,1,3) tổng  5: 2 ước                +  0 -> ans=0
(1,2,2) tổng  5: b=c x3               +  6 -> ans=6
(1,2,3) tổng  6: 3 ước                +  0 -> ans=6
(1,3,3) tổng  7: b=c x3               + 18 -> ans=24
(2,2,2) tổng  6: a=b=c (luôn 3 ước)   +  0 -> ans=24
(2,2,3) tổng  7: 0 ước                +  0 -> ans=24
(2,3,3) tổng  8: b=c x3               + 36 -> ans=60
(3,3,3) tổng  9: a=b=c (luôn 3 ước)   +  0 -> ans=60
Đáp án: 60 | brute force: 60


In [16]:
def trace_ordered_example1(nums):
    cnt = Counter(nums); total = 0
    for a, b, c_ in product(sorted(cnt), repeat=3):
        if single(a, b, c_) and ways(cnt, a, b, c_):
            w = ways(cnt, a, b, c_); total += w
            print(f"bộ giá trị có thứ tự ({a},{b},{c_}): tổng {a + b + c_}, ways = {w}, cộng dồn {total}")
    return total

print("Đáp án:", trace_ordered_example1([4, 6, 7, 3, 2]))

bộ giá trị có thứ tự (2,3,4): tổng 9, ways = 1, cộng dồn 1
bộ giá trị có thứ tự (2,4,3): tổng 9, ways = 1, cộng dồn 2
bộ giá trị có thứ tự (3,2,4): tổng 9, ways = 1, cộng dồn 3
bộ giá trị có thứ tự (3,4,2): tổng 9, ways = 1, cộng dồn 4
bộ giá trị có thứ tự (3,4,7): tổng 14, ways = 1, cộng dồn 5
bộ giá trị có thứ tự (3,7,4): tổng 14, ways = 1, cộng dồn 6
bộ giá trị có thứ tự (4,2,3): tổng 9, ways = 1, cộng dồn 7
bộ giá trị có thứ tự (4,3,2): tổng 9, ways = 1, cộng dồn 8
bộ giá trị có thứ tự (4,3,7): tổng 14, ways = 1, cộng dồn 9
bộ giá trị có thứ tự (4,7,3): tổng 14, ways = 1, cộng dồn 10
bộ giá trị có thứ tự (7,3,4): tổng 14, ways = 1, cộng dồn 11
bộ giá trị có thứ tự (7,4,3): tổng 14, ways = 1, cộng dồn 12
Đáp án: 12


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def singleDivisorTriplet(self, nums: List[int]) -> int:
        cnt = Counter(nums)
        vals = sorted(cnt)                      # tối đa 100 giá trị khác nhau
        ans = 0
        for a, b, c in combinations_with_replacement(vals, 3):   # a <= b <= c
            if a == c:
                continue                        # a = b = c: tổng 3a chia hết cho cả ba
            s = a + b + c
            if (s % a == 0) + (s % b == 0) + (s % c == 0) != 1:
                continue
            if a < b < c:
                ans += 6 * cnt[a] * cnt[b] * cnt[c]
            elif a == b:
                ans += 3 * cnt[a] * (cnt[a] - 1) * cnt[c]
            else:
                ans += 3 * cnt[b] * (cnt[b] - 1) * cnt[a]
        return ans

for nums, exp in examples:
    assert Solution().singleDivisorTriplet(nums) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Phân hoạch theo bộ giá trị.** Mỗi bộ chỉ số có thứ tự `(i, j, k)` ứng với đúng một bộ giá trị có thứ tự `(nums[i], nums[j], nums[k])`, và tính "một ước" chỉ phụ thuộc bộ giá trị đó (đếm theo vị trí nhưng giá trị ở mỗi vị trí đã xác định). Vậy đáp án `= Σ_{bộ giá trị thỏa} (số bộ chỉ số có bộ giá trị đó)`.

**Đếm số bộ chỉ số (Cách 3).** Chọn lần lượt `i` (có `c[a]` lựa chọn), `j ≠ i` với `nums[j] = b` (có `c[b] − [a=b]` lựa chọn vì nếu `b = a` thì `i` đã chiếm một vị trí giá trị `a`), `k` khác `i, j` với `nums[k] = c` (trừ các vị trí đã dùng có giá trị `c`). Quy tắc nhân cho đúng số bộ.

**Gộp theo dạng sắp xếp (Cách 4).** Các bộ giá trị có thứ tự cùng một đa tập `{a ≤ b ≤ c}` đều có cùng tính "một ước" (điều kiện đối xứng) và cùng số bộ chỉ số mỗi hoán vị. Với ba giá trị khác nhau: 6 hoán vị × `c[a]c[b]c[c]`. Với `a = b < c`: 3 hoán vị (vị trí của `c`) × `c[a](c[a]−1)c[c]`. Tương tự `a < b = c`. Với `a = b = c`: tổng `3a` chia hết cho cả ba ⇒ không bao giờ thỏa ⇒ bỏ qua đúng.

Ô dưới kiểm chứng bằng máy rằng với mọi bộ `a ≤ b ≤ c` trong 1..30, tổng `ways` trên mọi hoán vị bằng hệ số Cách 4 dùng.

In [18]:
rng = random.Random(10)
for _ in range(200):
    cnt = Counter({v: rng.randint(1, 4) for v in rng.sample(range(1, 31), 5)})
    for a, b, c_ in combinations_with_replacement(sorted(cnt), 3):
        total = sum(ways(cnt, *p) for p in set(permutations((a, b, c_))))
        if a < b < c_: coef = 6 * cnt[a] * cnt[b] * cnt[c_]
        elif a == b == c_: coef = cnt[a] * (cnt[a] - 1) * (cnt[a] - 2)
        elif a == b: coef = 3 * cnt[a] * (cnt[a] - 1) * cnt[c_]
        else: coef = 3 * cnt[b] * (cnt[b] - 1) * cnt[a]
        assert total == coef
print("Hệ số hoán vị của Cách 4 khớp tổng ways trên mọi hoán vị")
print("Bộ a=b=c có bao giờ 'single'? ->", any(single(a, a, a) for a in range(1, 101)))

Hệ số hoán vị của Cách 4 khớp tổng ways trên mọi hoán vị
Bộ a=b=c có bao giờ 'single'? -> False


## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `K` = số giá trị khác nhau (`K ≤ 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi bộ có thứ tự | `O(n³)` | `O(1)` |
| 2. `combinations` × 6 | `O(n³/6)` | `O(1)` |
| 3. Giá trị có thứ tự | `O(n + K³)` ≤ `10⁶` | `O(K)` |
| 4. Giá trị không giảm | `O(n + K³/6)` ≈ `1,7·10⁵` | `O(K)` |
| 5. Mảng đếm 1..100 | `O(n + K³)` | `O(100)` |

Đáp án có thể rất lớn (tới cỡ `n³`), Python không tràn số; trên LeetCode đáp án vừa `long`.

In [19]:
rng = random.Random(2198)
big = [rng.randint(1, 100) for _ in range(10**5)]
for name, fn in [("ordered", triplets_values_ordered), ("sorted", triplets_values_sorted), ("array", triplets_array),
                 ("Solution", lambda a: Solution().singleDivisorTriplet(a))]:
    t = time.perf_counter(); r = fn(big)
    print(f"{name:9s}: {r} ({time.perf_counter() - t:.3f}s)")

ordered  : 134085561125592 (0.107s)


sorted   : 134085561125592 (0.022s)


array    : 134085561125592 (0.080s)
Solution : 134085561125592 (0.018s)


## 12. Bộ test

- Ví dụ đề.
- Biên: `n = 3`; mọi phần tử bằng nhau (đáp án 0); có số 1 (1 luôn chia hết mọi tổng); hai giá trị bằng nhau chia hết tổng, giá trị còn lại không (phải loại).
- Đối chiếu mọi cách với brute force trên mảng ngẫu nhiên nhỏ (giá trị trong tập nhỏ để có nhiều lặp; seed cố định).

In [20]:
fns = {
    "brute": triplets_brute, "combinations": triplets_combinations, "ordered": triplets_values_ordered,
    "sorted": triplets_values_sorted, "array": triplets_array,
    "Solution": lambda a: Solution().singleDivisorTriplet(a), "SolutionOrdered": lambda a: SolutionOrdered().singleDivisorTriplet(a),
}
edge = [([5, 5, 5, 5], 0), ([2, 2, 6], 0), ([1, 2, 4], 6), ([2, 3, 4], 6), ([1, 100, 100], 6), ([3, 3, 3, 2], 18), ([1, 1, 1, 7], None)]
edge = [(a, triplets_brute(a)) if e is None else (a, e) for a, e in edge]
for nums, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(nums) == exp, (name, nums, fn(nums), exp)
print("Ví dụ + biên: OK", [(a, e) for a, e in edge])

Ví dụ + biên: OK [([5, 5, 5, 5], 0), ([2, 2, 6], 0), ([1, 2, 4], 6), ([2, 3, 4], 6), ([1, 100, 100], 6), ([3, 3, 3, 2], 18), ([1, 1, 1, 7], 0)]


In [21]:
rng = random.Random(42)
for _ in range(300):
    pool = rng.sample(range(1, 101), rng.randint(1, 6))
    nums = [rng.choice(pool) for _ in range(rng.randint(3, 12))]
    want = triplets_brute(nums)
    for name, fn in fns.items():
        assert fn(nums) == want, (name, nums)
for _ in range(100):
    nums = [rng.randint(1, 12) for _ in range(rng.randint(3, 25))]
    assert triplets_values_sorted(nums) == triplets_combinations(nums)
print("400 test ngẫu nhiên: mọi cách khớp brute force")

400 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm ước theo **giá trị khác nhau** (dùng `set`) thay vì theo vị trí

`[2, 2, 6]`: tổng `10` chia hết cho **hai vị trí** giá trị 2 ⇒ hai ước ⇒ **không** phải bộ một ước. Dùng `set` chỉ thấy "giá trị 2" một lần ⇒ đếm nhầm.

In [22]:
def wrong_set_divisors(nums):
    ans = 0
    for i, j, k in permutations(range(len(nums)), 3):
        vals = {nums[i], nums[j], nums[k]}                   # SAI: gộp các vị trí cùng giá trị
        s = nums[i] + nums[j] + nums[k]
        ans += sum(s % v == 0 for v in vals) == 1
    return ans

a = [2, 2, 6]
print("sai:", wrong_set_divisors(a), "| đúng:", triplets_brute(a))

sai: 6 | đúng: 0


In [23]:
print("sửa (đếm theo từng vị trí):", Solution().singleDivisorTriplet(a))

sửa (đếm theo từng vị trí): 0


### Lỗi 2 — Đếm bộ **không thứ tự** (quên nhân hoán vị)

Ví dụ 1 sẽ ra `2` thay vì `12`.

In [24]:
def wrong_unordered(nums):
    return sum(1 for i, j, k in combinations(range(len(nums)), 3) if single(nums[i], nums[j], nums[k]))   # SAI: thiếu x6

print("sai:", wrong_unordered([4, 6, 7, 3, 2]), "| đúng:", 12)

sai: 2 | đúng: 12


In [25]:
print("sửa (x6):", triplets_combinations([4, 6, 7, 3, 2]))

sửa (x6): 12


### Lỗi 3 — Nhân `cnt[a]·cnt[b]·cnt[c]` mà không trừ các vị trí trùng

Khi bộ giá trị có hai số bằng nhau, cùng một vị trí bị dùng hai lần ⇒ đếm thừa. Ví dụ 2 `[1, 2, 2]`.

In [26]:
def wrong_no_subtract(nums):
    cnt = Counter(nums); ans = 0
    for a, b, c_ in product(cnt, repeat=3):
        if single(a, b, c_):
            ans += cnt[a] * cnt[b] * cnt[c_]                 # SAI: không trừ trùng chỉ số
    return ans

print("sai:", wrong_no_subtract([1, 2, 2]), "| đúng:", 6)

sai: 12 | đúng: 6


In [27]:
print("sửa:", triplets_values_ordered([1, 2, 2]))

sửa: 6


### Lỗi 4 — Hiểu ngược phép chia: kiểm `nums[i] % tổng == 0`

"Tổng chia hết cho `nums[i]`" nghĩa là `tổng % nums[i] == 0`. Viết ngược `nums[i] % tổng` (luôn bằng `nums[i]` vì `nums[i] < tổng`) ⇒ không bao giờ đúng ⇒ đáp án 0.

In [28]:
def wrong_reversed_mod(nums):
    ans = 0
    for i, j, k in permutations(range(len(nums)), 3):
        s = nums[i] + nums[j] + nums[k]
        ans += (nums[i] % s == 0) + (nums[j] % s == 0) + (nums[k] % s == 0) == 1   # SAI: đảo chiều
    return ans

print("sai:", wrong_reversed_mod([4, 6, 7, 3, 2]), "| đúng:", 12)

sai: 0 | đúng: 12


In [29]:
print("sửa:", triplets_brute([4, 6, 7, 3, 2]))

sửa: 12


### Lỗi 5 — Ở Cách 4, dùng `C(cnt, 2)` cho trường hợp hai giá trị bằng nhau nhưng quên nhân 6 (hoặc nhân sai số hoán vị)

Với `a = b < c`, số bộ chỉ số có thứ tự là `6 · C(c[a], 2) · c[c] = 3 · c[a](c[a]−1) · c[c]`. Nhân 3 với `C(c[a], 2)` (thiếu một nửa) là lỗi hay gặp.

In [30]:
def wrong_perm_factor(nums):
    cnt = Counter(nums); ans = 0
    for a, b, c_ in combinations_with_replacement(sorted(cnt), 3):
        if a == c_ or not single(a, b, c_): continue
        if a < b < c_: ans += 6 * cnt[a] * cnt[b] * cnt[c_]
        elif a == b: ans += 3 * comb(cnt[a], 2) * cnt[c_]           # SAI: phải là 6 * C(.,2)
        else: ans += 3 * comb(cnt[b], 2) * cnt[a]
    return ans

print("sai:", wrong_perm_factor([1, 2, 2]), "| đúng:", 6)

sai: 3 | đúng: 6


In [31]:
print("sửa:", triplets_values_sorted([1, 2, 2]))

sửa: 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **`n` lớn nhưng giá trị nhỏ** (ở đây ≤ 100) ⇒ đếm theo **giá trị** bằng `Counter` / mảng đếm; độ phức tạp theo `K`, không theo `n`.
- **Bộ chỉ số có thứ tự, đôi một khác nhau** ⇒ quy tắc nhân **trừ trùng** `c[a]·(c[b]−[a=b])·...`.
- Điều kiện **đối xứng** ⇒ duyệt dạng sắp xếp `a ≤ b ≤ c` và nhân số hoán vị (6 / 3 / 1).
- "Đúng một trong ba" ⇒ cộng các boolean.
- Luôn kiểm trường hợp **giá trị trùng** bằng tay: đề đếm theo vị trí.

## 15. Biến thể của bài

1. **Bộ ba "ba ước"** (tổng chia hết cho cả ba vị trí) — bài tập 5.
2. **Đếm bộ không thứ tự** `i < j < k`: chia đáp án cho 6.
3. **Bộ bốn** với điều kiện tương tự: duyệt `K⁴/24` bộ giá trị không giảm, số hoán vị theo đa tập.
4. **LeetCode 1577 — Number of Ways Where Square of Number Is Equal to Product of Two Numbers:** cũng đổi vòng lặp sang giá trị + Counter.
5. **LeetCode 2475 — Number of Unequal Triplets in Array:** bộ ba giá trị đôi một khác nhau — đếm bằng tiền tố số lượng.

Minh họa biến thể 5 (tự dựng, kiểm bằng brute force):

In [32]:
def unequal_triplets(nums):
    cnt = Counter(nums); n = len(nums); left = 0; ans = 0
    for v, c_ in cnt.items():                 # giá trị v ở "giữa": trái (đã duyệt) x v x phải (chưa duyệt)
        right = n - left - c_
        ans += left * c_ * right
        left += c_
    return ans

rng = random.Random(2475)
for _ in range(300):
    a = [rng.randint(1, 5) for _ in range(rng.randint(3, 12))]
    assert unequal_triplets(a) == sum(1 for i, j, k in combinations(range(len(a)), 3) if len({a[i], a[j], a[k]}) == 3)
print("LeetCode 2475 (bộ ba khác nhau, không thứ tự) khớp brute force; [4,4,2,4,3] ->", unequal_triplets([4, 4, 2, 4, 3]))

LeetCode 2475 (bộ ba khác nhau, không thứ tự) khớp brute force; [4,4,2,4,3] -> 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi bộ có thứ tự | brute force | `O(n³)` | `O(1)` | kiểm tra |
| 2. `combinations` × 6 | bộ không thứ tự | `O(n³/6)` | `O(1)` | kiểm tra nhanh hơn chút |
| 3. Giá trị có thứ tự | `ways` trừ trùng | `O(n + K³)` | `O(K)` | dễ viết, khó sai |
| 4. Giá trị không giảm | nhân số hoán vị | `O(n + K³/6)` | `O(K)` | **nộp bài**, nhanh nhất |
| 5. Mảng đếm 1..100 | như 3, không hash | `O(n + K³)` | `O(100)` | dịch sang Java/C++ |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `singleDivisorTriplet([2, 3, 4])` và `singleDivisorTriplet([1, 2, 3])`.
2. **Tính tay:** `nums = [1, 1, 2, 3]` — liệt kê bộ giá trị không giảm, kiểm "một ước", nhân hệ số.
3. **Điền TODO:** `count_index_triplets(cnt, a, b, c)` — số bộ chỉ số có thứ tự với bộ giá trị **có thứ tự** `(a, b, c)`.
4. **Tự viết lại:** viết lại Cách 4 không nhìn code.
5. **Mini challenge:** `triple_divisor_triplets(nums)` — đếm bộ có thứ tự mà tổng chia hết cho **cả ba** vị trí. Kiểm bằng brute force.

In [33]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_index_triplets(cnt, a, b, c):
    # TODO: quy tắc nhân có trừ trùng
    pass

In [34]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_single_divisor(nums):
    # TODO
    pass

In [35]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def triple_divisor_triplets(nums):
    # TODO: như Cách 3/4 nhưng điều kiện là số ước == 3
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `[2,3,4]`: tổng 9 chỉ chia hết cho 3 ⇒ 6. (2) `[1,2,3]`: tổng 6 chia hết cho cả 1, 2, 3 ⇒ 0. (3) Kiểm lại bằng `triplets_brute`.

**Bài 2.** (1) `cnt = {1:2, 2:1, 3:1}`. (2) Bộ `(1,1,2)` tổng 4: vị trí 1, 1, 2 đều chia hết ⇒ 3 ước ✗; `(1,1,3)` tổng 5: hai vị trí 1 ⇒ 2 ước ✗; `(1,2,3)` tổng 6: 3 ước ✗; `(1,1,...)`... (3) Đáp án `0` — vì có hai số 1, mọi bộ chứa hai số 1 đều có ≥ 2 ước, và bộ chứa một số 1 cùng 2, 3 thì 6 chia hết cho cả ba.

**Bài 3.** (1) `cnt[a]`. (2) Nhân `cnt[b] − (a == b)`. (3) Nhân `cnt[c] − (c == a) − (c == b)`, kẹp về 0 nếu âm.

**Bài 4.** (1) `combinations_with_replacement(sorted(cnt), 3)`. (2) Bỏ `a == c`. (3) Ba trường hợp hệ số 6, 3, 3.

**Bài 5.** (1) Bây giờ `a = b = c` **luôn** thỏa: hệ số `c[a](c[a]−1)(c[a]−2)`. (2) Các trường hợp khác dùng như Cách 4. (3) Hoặc dùng Cách 3 với điều kiện `== 3`.

## 19. Đáp án tham khảo

In [36]:
# Bài 1, 2
assert Solution().singleDivisorTriplet([2, 3, 4]) == 6 and Solution().singleDivisorTriplet([1, 2, 3]) == 0
assert Solution().singleDivisorTriplet([1, 1, 2, 3]) == triplets_brute([1, 1, 2, 3]) == 0
print("Bài 1: 6 và 0 | Bài 2: 0")
trace_sorted([1, 1, 2, 3])

Bài 1: 6 và 0 | Bài 2: 0
cnt = {1: 2, 2: 1, 3: 1}
(1,1,1) tổng  3: a=b=c (luôn 3 ước)   +  0 -> ans=0
(1,1,2) tổng  4: 3 ước                +  0 -> ans=0
(1,1,3) tổng  5: 2 ước                +  0 -> ans=0
(1,2,2) tổng  5: b=c x3               +  0 -> ans=0
(1,2,3) tổng  6: 3 ước                +  0 -> ans=0
(1,3,3) tổng  7: b=c x3               +  0 -> ans=0
(2,2,2) tổng  6: a=b=c (luôn 3 ước)   +  0 -> ans=0
(2,2,3) tổng  7: 0 ước                +  0 -> ans=0
(2,3,3) tổng  8: b=c x3               +  0 -> ans=0
(3,3,3) tổng  9: a=b=c (luôn 3 ước)   +  0 -> ans=0


0

In [37]:
# Bài 3, 4
def count_index_triplets_ans(cnt, a, b, c):
    return max(cnt[a] * (cnt[b] - (a == b)) * (cnt[c] - (c == a) - (c == b)), 0)

my_single_divisor_ans = triplets_values_sorted
rng = random.Random(3)
for _ in range(200):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(3, 9))]
    cnt = Counter(nums)
    for a, b, c in product(range(1, 7), repeat=3):
        direct = sum(1 for i, j, k in permutations(range(len(nums)), 3) if (nums[i], nums[j], nums[k]) == (a, b, c))
        assert count_index_triplets_ans(cnt, a, b, c) == direct
    assert my_single_divisor_ans(nums) == triplets_brute(nums)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [38]:
# Bài 5
def triple_divisor_triplets_ans(nums):
    cnt = Counter(nums); ans = 0
    for a, b, c in combinations_with_replacement(sorted(cnt), 3):
        s = a + b + c
        if s % a or s % b or s % c:
            continue
        if a == c: ans += cnt[a] * (cnt[a] - 1) * (cnt[a] - 2)
        elif a < b < c: ans += 6 * cnt[a] * cnt[b] * cnt[c]
        elif a == b: ans += 3 * cnt[a] * (cnt[a] - 1) * cnt[c]
        else: ans += 3 * cnt[b] * (cnt[b] - 1) * cnt[a]
    return ans

def triple_divisor_brute(nums):
    return sum(1 for i, j, k in permutations(range(len(nums)), 3)
               if all((nums[i] + nums[j] + nums[k]) % v == 0 for v in (nums[i], nums[j], nums[k])))

rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(1, 8) for _ in range(rng.randint(3, 10))]
    assert triple_divisor_triplets_ans(nums) == triple_divisor_brute(nums)
print("Bài 5: OK; [1,2,3,3,3] ->", triple_divisor_triplets_ans([1, 2, 3, 3, 3]))

Bài 5: OK; [1,2,3,3,3] -> 24


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm bộ ba chỉ số khi giá trị nhỏ":**

```python
cnt = Counter(nums)
for a, b, c in combinations_with_replacement(sorted(cnt), 3):
    if not ok(a, b, c): continue
    if a < b < c:   ans += 6 * cnt[a] * cnt[b] * cnt[c]
    elif a == b == c: ans += cnt[a] * (cnt[a]-1) * (cnt[a]-2)
    elif a == b:    ans += 3 * cnt[a] * (cnt[a]-1) * cnt[c]
    else:           ans += 3 * cnt[b] * (cnt[b]-1) * cnt[a]
```

In [39]:
def template(nums, ok=single):
    cnt = Counter(nums); ans = 0
    for a, b, c in combinations_with_replacement(sorted(cnt), 3):
        if not ok(a, b, c): continue
        if a < b < c: ans += 6 * cnt[a] * cnt[b] * cnt[c]
        elif a == b == c: ans += cnt[a] * (cnt[a] - 1) * (cnt[a] - 2)
        elif a == b: ans += 3 * cnt[a] * (cnt[a] - 1) * cnt[c]
        else: ans += 3 * cnt[b] * (cnt[b] - 1) * cnt[a]
    return ans
print([template(a) for a, _ in examples])

[12, 6, 0]


**Checklist:**

- [ ] Đếm bộ **có thứ tự** (nhân hoán vị).
- [ ] "Đúng một ước" đếm theo **vị trí** (không dùng `set`).
- [ ] Kiểm `tổng % nums[x] == 0` (không đảo chiều).
- [ ] Trừ trùng chỉ số khi giá trị lặp: `c[a]·(c[b]−[a=b])·(c[c]−[c=a]−[c=b])`.
- [ ] Bỏ trường hợp `a = b = c` (luôn 3 ước).
- [ ] Duyệt theo **giá trị** (≤ 100), không theo chỉ số.